In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv



In [2]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [3]:
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    temperature=0.3,
    max_new_tokens=300,
)
llm = ChatHuggingFace(llm=endpoint)

In [6]:
class llm_state(TypedDict):
    question: str
    answer: str


In [10]:
def llm_qa(state: llm_state)-> llm_state:
    # extract the question from the state 
    question = state['question']

    # make prompt

    prompt = f'Answer the following question {question}'

    #ask the question from the llm

    answer = llm.invoke(prompt).content

    # update answer in the state

    state['answer'] = answer

    return state


In [13]:
#create graph 
graph = StateGraph(llm_state)

graph.add_node('llm_qa', llm_qa)

graph.add_edge(START,'llm_qa')

graph.add_edge('llm_qa',END)

workflow=graph.compile()

In [14]:
# execute
initial_state= {'question': 'How far is mars from earth?'}
final_state=workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is mars from earth?', 'answer': 'The distance between Earth and Mars varies significantly because both planets have elliptical orbits around the Sun. At their closest approach, known as opposition, Mars can be as close as about 54.6 million kilometers (33.9 million miles) from Earth. However, at their furthest, it can be as far as about 401 million kilometers (249 million miles) apart. On average, the distance between Earth and Mars is approximately 225 million kilometers (140 million miles).'}
